# Phase 3 — cleaning and seasonal features

Runs entirely offline from `data/processed/block_timeseries.parquet` (no
Earth Engine). Pipeline:

1. **Rolling-median outlier screen** (`vigor.clean`). Wildfire smoke and haze
   depress NDVI without tripping the Cloud Score+ mask; such points sit far
   below their local rolling median and are dropped. The screen is
   downward-only and runs per block-season.
2. **Savitzky-Golay smoothing**. Survivors are interpolated onto a daily grid
   (clipped to the span of real observations, so an in-progress season isn't
   extrapolated) and smoothed into one continuous curve per block-season.
3. **Per-season per-block features** (`vigor.features`): peak NDVI, the
   May–September integral, and the green-up date.

Features land in `data/processed/season_features.parquet`; per-block cleaning
figures go to `outputs/figures/03_{block_id}_cleaned_season.png`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from vigor import clean, features, ingest, plots

PARQUET = PROJECT_ROOT / "data" / "processed" / "block_timeseries.parquet"
FEATURES_PARQUET = PROJECT_ROOT / "data" / "processed" / "season_features.parquet"
FIGURES = PROJECT_ROOT / "outputs" / "figures"

In [ ]:
ts = ingest.load_timeseries(PARQUET)
print(f"{len(ts)} rows, {ts['date'].min().date()} to {ts['date'].max().date()}, "
      f"blocks: {sorted(ts['block_id'].unique())}")

## Clean and smooth

`clean_and_smooth` returns the screened observations (with an `is_outlier`
flag, for plotting what was removed) and the daily smoothed curve.

In [ ]:
observations, smoothed = clean.clean_and_smooth(ts)

n_flagged = int(observations['is_outlier'].sum())
print(f"seasonal observations: {len(observations)}  "
      f"flagged as smoke/haze: {n_flagged} ({100*n_flagged/len(observations):.1f}%)")
print(f"smoothed daily rows:   {len(smoothed)}\n")

dropped = observations[observations['is_outlier']]
print('dropped points by month (expect smoke in Aug 2021 / 2023, and 2025):')
dropped.assign(month=dropped['date'].dt.to_period('M')).groupby('month').size()

## Seasonal features

One row per (block, year): peak NDVI and its day of year, the May–Sep
integral (NDVI·days; NaN for a season that doesn't span the whole window),
and green-up date (half-max of the base→peak rise; NaN when a season has no
real green-up amplitude).

In [ ]:
feats = features.seasonal_features(smoothed)
features.write_features(feats, FEATURES_PARQUET)
print(f"features -> {FEATURES_PARQUET}\n")
feats

## Cleaning & feature figures

One panel per season per block: grey dots are kept observations, red ×'s are
dropped smoke/haze, the blue line is the Savitzky-Golay curve, the filled dot
is peak NDVI, and the dashed green rule is green-up.

In [ ]:
for block_id in sorted(smoothed['block_id'].unique()):
    fig = plots.cleaning_seasons(observations, smoothed, feats, block_id)
    saved = plots.save_figure(fig, FIGURES / f"03_{block_id}_cleaned_season.png")
    print(f"figure -> {saved}")

## Freeze cross-check

Peak NDVI per block per year. 2024 (known January freeze damage) should sit
low against each block's own history — the same signal Phase 2 flagged, now
on the cleaned/smoothed curve.

In [ ]:
feats.pivot(index='block_id', columns='year', values='peak_ndvi').round(3)